In [0]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1500

areas = ["Fiscal", "Faturamento", "Logística", "Compras", "Facilities", "TI"]
area_p = [0.18, 0.17, 0.20, 0.15, 0.12, 0.18]
categorias = ["Documento Fiscal", "Cadastro", "Requisição", "Suporte Sistema", "Logística", "Contrato"]
cat_p = [0.22, 0.16, 0.18, 0.17, 0.15, 0.12]
prioridades = ["Alta", "Média", "Baixa"]
prio_p = [0.25, 0.50, 0.25]

sla_map = {"Alta": 24, "Média": 48, "Baixa": 72}
media_area = {"Fiscal": 14, "Faturamento": 12, "Logística": 18, "Compras": 25, "Facilities": 16, "TI": 10}
fator_categoria = {"Documento Fiscal": 1.00, "Cadastro": 1.25, "Requisição": 1.15, "Suporte Sistema": 1.35, "Logística": 1.05, "Contrato": 1.10}
fator_prioridade = {"Alta": 0.80, "Média": 1.00, "Baixa": 1.30}

data_corte = pd.Timestamp("2026-10-31 18:00")

abertura = pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 300 * 24 * 60, n), unit="m")

df = pd.DataFrame({
    "id_chamado": [f"CH-{i:05d}" for i in range(1, n + 1)],
    "data_abertura": abertura,
    "area": rng.choice(areas, n, p=area_p),
    "categoria": rng.choice(categorias, n, p=cat_p),
    "prioridade": rng.choice(prioridades, n, p=prio_p),
})

df["sla_horas"] = df["prioridade"].map(sla_map)

media = (
    df["area"].map(media_area)
    * df["categoria"].map(fator_categoria)
    * df["prioridade"].map(fator_prioridade)
)
df["tempo_atendimento_horas"] = np.round(rng.lognormal(np.log(media), 0.45), 1)

em_andamento = rng.random(n) < 0.08
df["data_conclusao"] = df["data_abertura"] + pd.to_timedelta(df["tempo_atendimento_horas"], unit="h")
df.loc[em_andamento, "data_conclusao"] = pd.NaT
df.loc[em_andamento, "tempo_atendimento_horas"] = np.nan
df["status"] = np.where(em_andamento, "Em andamento", "Concluído")

horas_decorridas = (data_corte - df["data_abertura"]).dt.total_seconds() / 3600
df["sla_status"] = np.select(
    [
        df["status"].eq("Concluído") & (df["tempo_atendimento_horas"] <= df["sla_horas"]),
        df["status"].eq("Concluído"),
        horas_decorridas > df["sla_horas"],
    ],
    ["Cumprido", "Vencido", "Vencido"],
    default="No prazo",
)

df = df[["id_chamado", "data_abertura", "data_conclusao", "area", "categoria",
         "prioridade", "sla_horas", "tempo_atendimento_horas", "status", "sla_status"]]

display(df.head(10))
print(df.shape)

id_chamado,data_abertura,data_conclusao,area,categoria,prioridade,sla_horas,tempo_atendimento_horas,status,sla_status
CH-00001,2026-01-27T18:36:00.000Z,2026-01-28T15:06:00.000Z,Faturamento,Requisição,Alta,24,20.5,Concluído,Cumprido
CH-00002,2026-08-21T04:29:00.000Z,2026-08-21T18:41:00.000Z,Faturamento,Logística,Média,48,14.2,Concluído,Cumprido
CH-00003,2026-07-16T08:54:00.000Z,2026-07-17T00:30:00.000Z,Faturamento,Cadastro,Média,48,15.6,Concluído,Cumprido
CH-00004,2026-05-12T15:55:00.000Z,2026-05-13T12:37:00.000Z,Logística,Suporte Sistema,Baixa,72,20.7,Concluído,Cumprido
CH-00005,2026-05-10T21:42:00.000Z,2026-05-12T01:18:00.000Z,Compras,Contrato,Média,48,27.6,Concluído,Cumprido
CH-00006,2026-09-15T13:54:00.000Z,2026-09-16T03:48:00.000Z,Faturamento,Requisição,Média,48,13.9,Concluído,Cumprido
CH-00007,2026-01-26T18:48:00.000Z,2026-01-27T01:54:00.000Z,Faturamento,Contrato,Média,48,7.1,Concluído,Cumprido
CH-00008,2026-07-29T05:02:00.000Z,2026-07-29T13:14:00.000Z,TI,Logística,Média,48,8.2,Concluído,Cumprido
CH-00009,2026-03-02T10:34:00.000Z,2026-03-03T02:16:00.000Z,Fiscal,Documento Fiscal,Baixa,72,15.7,Concluído,Cumprido
CH-00010,2026-01-29T06:04:00.000Z,2026-01-30T01:58:00.000Z,Logística,Requisição,Alta,24,19.9,Concluído,Cumprido


(1500, 10)
